In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris()
dane = iris.data
target = iris.target
nazwy_gatunkow = iris.target_names

minimum = np.min(dane, axis=0)
maksimum = np.max(dane, axis=0)
dane_znormalizowane = (dane - minimum) / (maksimum - minimum)

print("Dane przygotowane. Kształt:", dane_znormalizowane.shape)

Dane przygotowane. Kształt: (150, 4)


In [2]:
def trenuj_som(dane, epoki=50, poczatkowy_wspolczynnik=0.1, funkcja_spadku='liniowa'):
    np.random.seed(42)
    wagi = np.random.rand(3, 4)

    T = epoki
    alfa_0 = poczatkowy_wspolczynnik

    print(f"--- Trening rozpoczęty ({epoki} epok, funkcja: {funkcja_spadku}) ---")

    for t in range(T):
        if funkcja_spadku == 'liniowa':
            wspolczynnik = alfa_0 * (1 - t / T)
        elif funkcja_spadku == 'wykladnicza':
            wspolczynnik = alfa_0 * np.exp(-t / (T / 4))

        if t % 10 == 0:
            print(f"Epoka {t}: współczynnik uczenia = {wspolczynnik:.4f}")

        for wektor in dane:
            odleglosci = np.sqrt(np.sum((wagi - wektor) ** 2, axis=1))
            bmu_idx = np.argmin(odleglosci)
            wagi[bmu_idx] += wspolczynnik * (wektor - wagi[bmu_idx])

    print("--- Koniec treningu ---\n")
    return wagi

In [3]:
def przypisz_i_ocen(wagi, dane, target, nazwy_gatunkow):
    przypisania = []

    for wektor in dane:
        odleglosci = np.sqrt(np.sum((wagi - wektor) ** 2, axis=1))
        przypisania.append(np.argmin(odleglosci))

    df = pd.DataFrame({
        'Neuron': [f"N{i + 1}" for i in przypisania],
        'Gatunek': [nazwy_gatunkow[i].capitalize() for i in target]
    })

    macierz = pd.crosstab(df['Neuron'], df['Gatunek'])

    for gatunek in [n.capitalize() for n in nazwy_gatunkow]:
        if gatunek not in macierz.columns:
            macierz[gatunek] = 0

    macierz = macierz[[n.capitalize() for n in nazwy_gatunkow]]  # sortowanie kolumn

    print("Macierz przypisań przykładów do neuronów:")
    display(macierz)
    return przypisania

In [4]:
wagi_koncowe = trenuj_som(dane_znormalizowane, epoki=50, poczatkowy_wspolczynnik=0.1)

print("Końcowe wagi 3 neuronów:")
print(np.round(wagi_koncowe, 4))
print("\n")

_ = przypisz_i_ocen(wagi_koncowe, dane_znormalizowane, target, nazwy_gatunkow)

--- Trening rozpoczęty (50 epok, funkcja: liniowa) ---
Epoka 0: współczynnik uczenia = 0.1000
Epoka 10: współczynnik uczenia = 0.0800
Epoka 20: współczynnik uczenia = 0.0600
Epoka 30: współczynnik uczenia = 0.0400
Epoka 40: współczynnik uczenia = 0.0200
--- Koniec treningu ---

Końcowe wagi 3 neuronów:
[[0.5595 0.3715 0.6784 0.6796]
 [0.1949 0.5838 0.0859 0.0688]
 [0.6011 0.7081 0.0206 0.9699]]


Macierz przypisań przykładów do neuronów:


Gatunek,Setosa,Versicolor,Virginica
Neuron,,,
N1,0,49,50
N2,50,1,0


In [5]:
print("================ EKSPERYMENT 1: 200 epok, liniowy spadek ================")
wagi_eks1 = trenuj_som(dane_znormalizowane, epoki=200, poczatkowy_wspolczynnik=0.5, funkcja_spadku='liniowa')
_ = przypisz_i_ocen(wagi_eks1, dane_znormalizowane, target, nazwy_gatunkow)

print("\n================ EKSPERYMENT 2: 50 epok, spadek wykładniczy ================")
wagi_eks2 = trenuj_som(dane_znormalizowane, epoki=50, poczatkowy_wspolczynnik=0.2, funkcja_spadku='wykladnicza')
_ = przypisz_i_ocen(wagi_eks2, dane_znormalizowane, target, nazwy_gatunkow)

print("\n================ EKSPERYMENT 3: 5 epok (zbyt mało nauki) ================")
wagi_eks3 = trenuj_som(dane_znormalizowane, epoki=5, poczatkowy_wspolczynnik=0.1, funkcja_spadku='liniowa')
_ = przypisz_i_ocen(wagi_eks3, dane_znormalizowane, target, nazwy_gatunkow)

================ EKSPERYMENT 1: 200 epok, liniowy spadek ================
--- Trening rozpoczęty (200 epok, funkcja: liniowa) ---
Epoka 0: współczynnik uczenia = 0.5000
Epoka 10: współczynnik uczenia = 0.4750
Epoka 20: współczynnik uczenia = 0.4500
Epoka 30: współczynnik uczenia = 0.4250
Epoka 40: współczynnik uczenia = 0.4000
Epoka 50: współczynnik uczenia = 0.3750
Epoka 60: współczynnik uczenia = 0.3500
Epoka 70: współczynnik uczenia = 0.3250
Epoka 80: współczynnik uczenia = 0.3000
Epoka 90: współczynnik uczenia = 0.2750
Epoka 100: współczynnik uczenia = 0.2500
Epoka 110: współczynnik uczenia = 0.2250
Epoka 120: współczynnik uczenia = 0.2000
Epoka 130: współczynnik uczenia = 0.1750
Epoka 140: współczynnik uczenia = 0.1500
Epoka 150: współczynnik uczenia = 0.1250
Epoka 160: współczynnik uczenia = 0.1000
Epoka 170: współczynnik uczenia = 0.0750
Epoka 180: współczynnik uczenia = 0.0500
Epoka 190: współczynnik uczenia = 0.0250
--- Koniec treningu ---

Macierz przypisań przykładów do neur

Gatunek,Setosa,Versicolor,Virginica
Neuron,,,
N1,50,0,0
N2,0,47,14
N3,0,3,36



================ EKSPERYMENT 2: 50 epok, spadek wykładniczy ================
--- Trening rozpoczęty (50 epok, funkcja: wykladnicza) ---
Epoka 0: współczynnik uczenia = 0.2000
Epoka 10: współczynnik uczenia = 0.0899
Epoka 20: współczynnik uczenia = 0.0404
Epoka 30: współczynnik uczenia = 0.0181
Epoka 40: współczynnik uczenia = 0.0082
--- Koniec treningu ---

Macierz przypisań przykładów do neuronów:


Gatunek,Setosa,Versicolor,Virginica
Neuron,,,
N1,50,0,0
N2,0,47,14
N3,0,3,36



================ EKSPERYMENT 3: 5 epok (zbyt mało nauki) ================
--- Trening rozpoczęty (5 epok, funkcja: liniowa) ---
Epoka 0: współczynnik uczenia = 0.1000
--- Koniec treningu ---

Macierz przypisań przykładów do neuronów:


Gatunek,Setosa,Versicolor,Virginica
Neuron,,,
N1,0,46,50
N2,50,4,0
